# Git, DVC, and Experiment Tracking

## Table of Contents

1. [Git Essentials for DS Projects](#1)
2. [What to .gitignore](#2)
3. [DVC: Version Control for Data and Models](#3)
4. [Experiment Tracking with MLflow](#4)
5. [Putting It Together](#5)

---
## 1. Git Essentials for DS Projects <a id="1"></a>

You should already know the basics: `git init`, `add`, `commit`, `push`, `pull`. If not, work through [learngitbranching.js.org](https://learngitbranching.js.org/) — it's excellent.

This notebook focuses on what changes for DS workflows.

### The commit that matters

Git tracks *code*. For a reproducible ML experiment, your commit message should record what you changed **and why the result changed**:

```
# ❌ Not useful
git commit -m "update model"

# ✅ Useful
git commit -m "feat: add log-tenure feature

AUC improved from 0.81 to 0.84 on churn holdout.
Feature captures the non-linear relationship between
tenure length and churn probability."
```

### Branch strategy

```
main           ← only working, reviewed code
├── dev        ← integration branch
│   ├── feature/add-log-tenure
│   ├── feature/pydantic-config
│   └── bugfix/missing-value-leak
```

For solo DS work, even just `main` + `feature/*` branches is enough to avoid committing broken code directly to main.

---
## 2. What to .gitignore <a id="2"></a>

This is the most DS-specific git skill. The wrong `.gitignore` means you accidentally commit 2GB of data or secret API keys.

A minimal `.gitignore` for a DS project:

```gitignore
# Python
__pycache__/
*.pyc
*.pyo
.venv/
dist/
build/
*.egg-info/

# Data — NEVER commit raw or processed data
data/raw/
data/processed/
data/interim/
*.csv
*.parquet
*.feather

# Models — large binary files
models/
*.pkl
*.joblib
*.pt
*.h5

# Jupyter
.ipynb_checkpoints/

# Secrets
.env
*.key
secrets.yaml

# IDE
.vscode/
.idea/
```

**Critical**: add `.gitignore` *before* your first commit. Removing a large file from git history after the fact is painful.

In [ ]:
# The repo already has a .gitignore — print it so we can discuss it
gitignore_path = "../.gitignore"
try:
    with open(gitignore_path) as f:
        print(f.read())
except FileNotFoundError:
    print(f"Not found at {gitignore_path} — run from the notebooks/ directory")

---
## 3. DVC: Version Control for Data and Models <a id="3"></a>

Git is for code. **DVC** (Data Version Control) is for data, models, and pipeline definitions. It integrates with git so your code commit and your data version are always in sync.

Install: `pip install dvc`

### The problem DVC solves

Without DVC:
- "Which dataset did we train v3 on?" — *no idea*
- "Reproduce last month's model" — *good luck*
- "The data changed — which experiments are stale?" — *nobody knows*

With DVC:
- Every git commit has a corresponding `.dvc` pointer file that identifies the exact dataset
- `dvc repro` re-runs the pipeline with the right data
- `dvc diff` shows what data changed

In [ ]:
# DVC quickstart (run these in your terminal, not here)
print("""# 1. Install and initialise
pip install dvc
git init
dvc init

# 2. Track your data directory (stores the actual data in a remote, not git)
dvc add data/raw/churn.csv
git add data/raw/churn.csv.dvc .gitignore
git commit -m "track raw churn data with DVC"

# 3. Set a remote storage (S3, GCS, Azure, or local)
dvc remote add -d myremote s3://my-bucket/dvc-store
dvc push

# 4. Now on another machine (or after checkout):
git clone <repo>
dvc pull   # downloads the exact data version from the remote
""")

print("What you get in the repo:")
print("  data/raw/churn.csv.dvc  <- tiny pointer file, committed to git")
print("  data/raw/churn.csv      <- actual data, in remote storage only")

### DVC Pipelines

DVC can also define and cache the stages of your pipeline, so `dvc repro` only re-runs stages whose inputs changed:

```yaml
# dvc.yaml
stages:
  prepare:
    cmd: python src/churn_project/data.py
    deps:
      - data/raw/churn.csv
      - src/churn_project/data.py
    outs:
      - data/processed/churn_clean.csv

  train:
    cmd: python src/churn_project/models/train.py
    deps:
      - data/processed/churn_clean.csv
      - src/churn_project/models/train.py
    outs:
      - models/rf_v1.pkl
    metrics:
      - metrics/scores.json
```

```bash
dvc repro        # runs only stages whose deps changed
dvc dag          # visualise the pipeline as a graph
dvc metrics show # compare metrics across commits
```

---
## 4. Experiment Tracking with MLflow <a id="4"></a>

DVC handles data and pipeline state. **MLflow** handles experiment tracking — logging every run's parameters, metrics, and artefacts so you can compare them and find the best model.

Install: `pip install mlflow`

In [ ]:
try:
    import mlflow
    import mlflow.sklearn
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, roc_auc_score

    # Set the tracking URI — where MLflow stores run data
    # Local filesystem is fine for solo work; use a server for teams
    mlflow.set_tracking_uri("file:///tmp/mlruns_demo")
    mlflow.set_experiment("churn_model")

    # Simulate some data
    np.random.seed(42)
    X = pd.DataFrame({"tenure": np.random.randint(1, 72, 300),
                      "charges": np.random.uniform(20, 100, 300)})
    y = pd.Series((np.random.rand(300) > 0.7).astype(int))
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    # Log a run
    with mlflow.start_run(run_name="baseline_rf"):
        # Log parameters
        params = {"n_estimators": 100, "max_depth": 5, "random_state": 42}
        mlflow.log_params(params)

        # Train
        clf = RandomForestClassifier(**params)
        clf.fit(X_tr, y_tr)

        # Log metrics
        acc = accuracy_score(y_te, clf.predict(X_te))
        auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
        mlflow.log_metrics({"accuracy": acc, "auc": auc})

        # Log the model itself
        mlflow.sklearn.log_model(clf, "random_forest")

        print(f"Run logged: accuracy={acc:.3f}, AUC={auc:.3f}")
        print(f"Run ID: {mlflow.active_run().info.run_id}")

except ImportError:
    print("Install MLflow: pip install mlflow")
    print()
    print("Key concepts:")
    print("  mlflow.set_experiment(\"my_experiment\")")
    print("  with mlflow.start_run():")
    print("      mlflow.log_params({...})      # hyperparameters")
    print("      mlflow.log_metrics({...})     # scores")
    print("      mlflow.sklearn.log_model(clf) # the model artefact")
    print()
    print("View results: mlflow ui  (then open http://localhost:5000)")

### What gets logged and why

| MLflow concept | What it stores | Example |
|---|---|---|
| **Params** | Hyperparameters and settings | `n_estimators=100`, `max_depth=5` |
| **Metrics** | Numeric scores (can be per-step) | `accuracy=0.87`, `auc=0.91` |
| **Artefacts** | Files: models, plots, configs | `model.pkl`, `confusion_matrix.png` |
| **Tags** | Free-form metadata | `"dataset": "churn_v2"`, `"author": "alice"` |

After logging a few runs, `mlflow ui` gives you a comparison table — find which hyperparameters improved AUC, reproduce any previous run exactly.

In [ ]:
try:
    import mlflow

    # Wrapping the whole pipeline run in a helper
    def run_experiment(
        X_train, X_test, y_train, y_test,
        n_estimators: int = 100,
        max_depth: int = None,
        run_name: str = "experiment",
    ) -> float:
        with mlflow.start_run(run_name=run_name):
            params = {"n_estimators": n_estimators, "max_depth": max_depth}
            mlflow.log_params(params)

            clf = RandomForestClassifier(**{k: v for k, v in params.items() if v is not None},
                                         random_state=42)
            clf.fit(X_train, y_train)

            acc = accuracy_score(y_test, clf.predict(X_test))
            auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
            mlflow.log_metrics({"accuracy": acc, "auc": auc})
            mlflow.sklearn.log_model(clf, "model")

            print(f"  {run_name}: acc={acc:.3f} auc={auc:.3f}")
            return auc

    mlflow.set_tracking_uri("file:///tmp/mlruns_demo")
    mlflow.set_experiment("churn_grid")

    print("Running grid search with MLflow tracking:")
    best_auc = 0.0
    for n_est in [50, 100, 200]:
        for depth in [3, 5, None]:
            auc = run_experiment(
                X_tr, X_te, y_tr, y_te,
                n_estimators=n_est,
                max_depth=depth,
                run_name=f"rf_n{n_est}_d{depth}",
            )
            best_auc = max(best_auc, auc)

    print(f"\nBest AUC: {best_auc:.3f}")
    print("View all runs: mlflow ui")

except ImportError:
    print("MLflow not installed — see cell above for concepts")

---
## 5. Putting It Together <a id="5"></a>

A reproducible DS project uses all three tools together:

```
Git     ← tracks code changes
DVC     ← tracks data and pipeline stage outputs
MLflow  ← tracks experiment results (params, metrics, models)
```

A typical workflow:

```bash
# 1. Start a feature branch
git checkout -b feature/log-tenure-feature

# 2. Change the feature engineering code
vim src/churn_project/features.py

# 3. Re-run the pipeline (DVC will only re-run changed stages)
dvc repro

# 4. Compare results in MLflow
mlflow ui
# → "log-tenure gave +0.03 AUC — ship it"

# 5. Commit code + DVC pointer files together
git add src/ data/processed.dvc dvc.lock metrics/
git commit -m "feat: log-tenure feature improves AUC by 0.03"

# 6. Push code and data separately
git push
dvc push
```

Now any colleague can:
```bash
git checkout feature/log-tenure-feature
dvc pull     # gets exactly the data that produced those results
dvc repro    # reproduces the pipeline
```

This is what reproducible ML looks like in practice.